# Generic Signal Generator (`SWGO_Gen_Ns`)

Computes **`gen_ns`** — the expected signal count rate **per unit J-factor and per unit cross-section** — for a chosen annihilation channel.

$$
\text{gen\_ns} = \frac{N_s}{\langle\sigma v\rangle \cdot J}
\quad\Longrightarrow\quad
N_s = \langle\sigma v\rangle \cdot J \cdot \text{gen\_ns}
$$

Because `gen_ns` does **not** depend on the specific source (i.e. it contains neither $\langle\sigma v\rangle$ nor $J$), it only needs to be computed **once per channel**.  
When analyzing a concrete dwarf galaxy, multiply by that object's J-factor to obtain the predicted signal counts.

The $\gamma$-ray photon flux from DM annihilation is:
$$
\frac{d\Phi}{dE} = \frac{\langle\sigma v\rangle}{8\pi\, m_{\rm DM}^2}\,\frac{dN}{dE}
\int_{\rm l.o.s.} ds \int d\Omega\; \rho_{\rm DM}^2
$$

Spectra (`dN/dE`) are taken from PPPC4DMID (`AtProduction_gammas.dat`).  
SWGO Instrument Response Functions (IRFs) are read from the `inputs/` folder:

| File | Content |
|------|---------|
| `hArea_swgo_i_g.txt` | Effective area vs energy |
| `swgo_BckRate_per_sr.txt` | Background rate per steradian vs energy |
| `SWGO_i_Edisp_g.txt` | Energy dispersion matrix |

Source: <https://github.com/harmscho/SGSOSensitivity>


## Imports

In [10]:
import numpy as np
from scipy import integrate
from scipy.interpolate import interp1d
from joblib import Parallel, delayed   # for parallelisation
import os


## ⚙️ Input Cell — configure everything here

All paths and physics parameters are set in this single cell.


In [11]:
# ── Annihilation channel ──────────────────────────────────────────────────────
# Available channels (column names from PPPC4DMID):
#   'eL','eR','e','muL','muR','mu','tauL','tauR','tau',
#   'q','c','b','t','WL','WT','W','ZL','ZT','Z','g','gamma','h',
#   'nue','numu','nutau','Ve','Vmu','Vtau'
CHANNEL = 'b'          # channel to compute gen_ns for

# ── Observation time ─────────────────────────────────────────────────────────
# 6 observing hours/day × 10 years ÷ 4  (duty-cycle factor)
TIME = (10 * 3600 * 24 * 365.25) / 4   # [s]

# ── Paths ─────────────────────────────────────────────────────────────────────
PPPC_FILE   = 'AtProduction_gammas.dat'   # PPPC4DMID spectrum table

# SWGO IRFs (from https://github.com/harmscho/SGSOSensitivity)
AREA_FILE   = 'inputs/hArea_swgo_i_g.txt'
BKG_FILE    = 'inputs/swgo_BckRate_per_sr.txt'
EDISP_FILE  = 'inputs/SWGO_i_Edisp_g.txt'

# Output
RESULTS_DIR = 'results/gen_ns'
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Parallelisation ───────────────────────────────────────────────────────────
N_JOBS = -1   # -1 → use all available CPU cores; set to 1 to disable


## Load & pre-process input data

In [12]:
# ── PPPC4DMID spectra ─────────────────────────────────────────────────────────
# Columns: mDM [GeV], log10(x), then dN/d(log10 x) for each channel.
# We convert to dN/dE [GeV^-1] using:
#     dN/dE = dN/d(log10 x) / (ln(10) · E)   with x = E/mDM
flux = np.loadtxt(PPPC_FILE, skiprows=1)

# Column 1 → E = 10^(log10 x) * mDM  ... already stored as log10(x)
# First: flux[:,0] = mDM [GeV], flux[:,1] = log10(x)
flux[:, 1] = (10 ** flux[:, 1]) * flux[:, 0]   # E [GeV]

# Remaining columns: convert dN/d(log10 x)  →  dN/dE [GeV^-1]
for i in range(2, 30):
    flux[:, i] *= 1.0 / (np.log(10) * flux[:, 1])

mass = np.unique(flux[:, 0])              # DM mass array [GeV]
np.savetxt('mass.txt', mass)              # cache for downstream notebooks

# Organise spectra by mass for fast look-up: shape = (E-points, columns, n_mass)
n_pts  = len(flux[flux[:, 0] == mass[0]])
flux_by_mass = np.zeros((n_pts, flux.shape[1], len(mass)))
for k, mk in enumerate(mass):
    flux_by_mass[:, :, k] = flux[flux[:, 0] == mk, :]

# Column index corresponding to the chosen channel
CHANNEL_NAMES = [
    'mDM','Log10x',
    'eL','eR','e','muL','muR','mu','tauL','tauR','tau',
    'q','c','b','t','WL','WT','W','ZL','ZT','Z',
    'g','gamma','h','nue','numu','nutau','Ve','Vmu','Vtau'
]
CHANNEL_IDX = CHANNEL_NAMES.index(CHANNEL)
print(f"Channel '{CHANNEL}' → column index {CHANNEL_IDX}")

# ── Effective area ─────────────────────────────────────────────────────────────
# File columns: log10(E/GeV),  Area [m²]
# We convert:  log10(E/GeV) → E [GeV]   and   m² → cm²  (×1e4)
eff_area = np.loadtxt(AREA_FILE, skiprows=1)
eff_area[:, 0] = 10 ** eff_area[:, 0]   # E [GeV]
eff_area[:, 1] *= 1e4                    # [cm²]

area = interp1d(eff_area[:, 0], eff_area[:, 1],
                bounds_error=False, fill_value=0)
print(f"Effective area loaded: {len(eff_area)} energy points")
print(f"  Energy range: {eff_area[0,0]:.1f} – {eff_area[-1,0]:.1f} GeV")

# ── Background rate ────────────────────────────────────────────────────────────
# File columns: E_min [TeV],  E_max [TeV],  BckRate [Hz/sr]
# We convert:  TeV → GeV  (×1e3)
# Rows with rate = 0 are discarded (SWGO sensitivity range).
# After multiplying by TIME [s]: bkg[:,2] = integrated background counts / sr.
background = np.loadtxt(BKG_FILE, skiprows=1)
background[:, 0] *= 1e3   # E_min  [GeV]
background[:, 1] *= 1e3   # E_max  [GeV]

bkg = background[background[:, 2] > 0].copy()
bkg[:, 2] *= TIME          # [counts / sr]  (Hz/sr × s)

np.savetxt('bkg.txt', bkg)  # cache for downstream notebooks
print(f"Background bins: {len(bkg)}")
print(f"  Energy range: {bkg[0,0]:.0f} – {bkg[-1,1]:.0f} GeV")

# ── Energy dispersion (convolution kernel) ─────────────────────────────────────
# File columns: Etrue_bin, Etrue, Etrue_min, Etrue_max,
#               Ereco_bin, Ereco, Ereco_min, Ereco_max, Edisp_value
# We group by reconstructed energy bin (Ereco_max) to build the kernel.
convolution_data = np.loadtxt(EDISP_FILE, skiprows=1)

energy_values  = np.unique(convolution_data[:, 1])   # true E bin centres [GeV]
ienergy_values = np.unique(convolution_data[:, 2])   # true E bin lower edges [GeV]
fenergy_values = np.unique(convolution_data[:, 3])   # true E bin upper edges [GeV]

n_ebins = len(fenergy_values)
n_rows_per_bin = len(convolution_data[convolution_data[:, 3] == fenergy_values[0]])

conv_by_range = np.zeros((n_ebins, n_rows_per_bin, convolution_data.shape[1]))
for k in range(n_ebins):
    conv_by_range[k] = convolution_data[convolution_data[:, 3] == fenergy_values[k]]

# Column 8 is Edisp_value; shape → (reco-E bins, true-E bins)
rate = conv_by_range[:, :, 8]

# Normalise each reconstructed-energy slice so it is a proper probability distribution.
# Rows where the total is 0 (outside sensitivity range) are left as-is.
for j in range(len(energy_values)):
    total = np.sum(rate[j])
    if total != 0:
        rate[j] *= 1.0 / total

print(f"Energy dispersion kernel: {rate.shape[0]}×{rate.shape[1]}")


Channel 'b' → column index 13
Effective area loaded: 162 energy points
  Energy range: 9.6 – 1036621.0 GeV
Background bins: 35
  Energy range: 237 – 100000 GeV
Energy dispersion kernel: 160×160


## Functions

### `intS` — signal integrand (per unit $J$ and $\langle\sigma v\rangle$)

$$
I(E) = \frac{1}{8\pi\, m_{\rm DM}^2}\; T\; \frac{dN}{dE}(E)\; A_{\rm eff}(E)
$$

| Argument | Unit | Description |
|----------|------|-------------|
| `E`      | GeV | photon energy |
| `T`      | s   | observation time |
| `dN`     | GeV⁻¹ | interpolated spectrum |
| `mass_k` | GeV | DM particle mass |

**Returns:** integrand in units $\text{cm}^2\,\text{s}\,\text{GeV}^{-2}$ so that  
$\int I\,dE$ has units $\text{cm}^2\,\text{s}\,\text{GeV}^{-1}$, and after  
multiplying by $\langle\sigma v\rangle\,[\text{cm}^3\text{s}^{-1}]\times J\,[\text{GeV}^2\text{cm}^{-5}]$ one obtains a dimensionless photon count.


In [13]:
def intS(E, T, dN, mass_k):
    """
    Integrand of the pre-J signal integral for a single DM mass.

    Parameters
    ----------
    E      : array_like [GeV]   photon energy values
    T      : float       [s]    observation time
    dN     : callable    [GeV^-1]  interpolated spectrum dN/dE(E)
    mass_k : float       [GeV]  DM particle mass

    Returns
    -------
    ndarray  [cm^2 s GeV^-2]  value of the integrand at each E
    """
    return (1.0 / (8.0 * np.pi * mass_k**2)) * T * dN(E) * area(E)


### `_signal_for_mass` — signal integral for one mass (parallelisable unit)

Integrates `intS` over each energy bin for a single DM mass index `k`.  
Returns a 1-D array of shape `(n_energy_bins,)`.

Energy bins where `E_min ≥ m_DM` contribute zero (kinematics).  
Bins that straddle `m_DM` are clipped at `m_DM` as the upper limit.


In [14]:
def _signal_for_mass(k):
    """
    Compute the signal integral for DM mass index k (used in Parallel loop).

    Integrates the intS over each true-energy bin.
    Bins entirely above m_DM are set to zero (kinematically forbidden).

    Parameters
    ----------
    k : int   index into the global `mass` array

    Returns
    -------
    s_k : ndarray, shape (n_energy_bins,)  [cm^2 s GeV^-1]
    """
    mass_k = mass[k]   # [GeV]
    dN = interp1d(
        flux_by_mass[:, 1, k],          # E axis  [GeV]
        flux_by_mass[:, CHANNEL_IDX, k], # dN/dE   [GeV^-1]
        bounds_error=False, fill_value=0
    )

    s_k = np.zeros(len(energy_values))
    for i in range(len(energy_values)):
        E_lo = ienergy_values[i]
        E_hi = fenergy_values[i]

        if E_hi <= mass_k:
            # Entire bin below m_DM — integrate normally
            quat = np.linspace(E_lo, E_hi, 10_000)
            s_k[i] = integrate.simpson(intS(quat, TIME, dN, mass_k), quat)

        elif E_lo < mass_k <= E_hi:
            # Bin straddles the kinematic cutoff — clip upper limit to m_DM
            quat = np.linspace(E_lo, mass_k, 10_000)
            s_k[i] = integrate.simpson(intS(quat, TIME, dN, mass_k), quat)

        # else: E_lo >= mass_k → s_k[i] stays 0

    return s_k


### `s_integral` — vectorised over all masses (with optional parallelisation)

Calls `_signal_for_mass` for every DM mass in parallel.


In [15]:
def s_integral():
    """
    Compute the signal integral for all DM masses in parallel.

    Returns
    -------
    s : ndarray, shape (n_energy_bins, n_mass)  [cm^2 s GeV^-1]
    """
    results = Parallel(n_jobs=N_JOBS)(
        delayed(_signal_for_mass)(k) for k in range(len(mass))
    )
    return np.column_stack(results)


### Energy-dispersion convolution

Smears the true-energy signal into reconstructed-energy bins using the SWGO
energy-dispersion matrix `rate`.


In [16]:
def find_suplimit(array, value):
    """
    Return the index of the first element in `array` that is ≥ `value`.
    Used to map background bin boundaries to the dispersion matrix grid.
    """
    array = np.asarray(array)
    i = np.abs(array - value).argmin()
    if value - array[i] <= 0:
        return i
    return i + 1


def find_inflimit(array, value):
    """
    Return the index of the last element in `array` that is ≤ `value`.
    Used to map background bin boundaries to the dispersion matrix grid.
    """
    array = np.asarray(array)
    i = np.abs(array - value).argmin()
    if value - array[i] >= 0:
        return i
    return i - 1


def apply_convolution(s_col):
    """
    Apply the energy-dispersion kernel to a single mass column of s.

    Parameters
    ----------
    s_col : ndarray, shape (n_energy_bins,)  true-energy signal for one mass

    Returns
    -------
    s_conv : ndarray, shape (n_reco_bins,)   reconstructed-energy signal
    """
    # rate[i, j] = P(reco bin j | true bin i)  — rows normalised to 1.
    # Forward-smearing: result[j] = Σ_i  s_col[i] * rate[i, j]  =  s_col @ rate
    #
    # s_col[:, None]  shape: (n_true, 1)
    # rate            shape: (n_true, n_reco)
    # product         shape: (n_true, n_reco)
    # sum over axis 0 →      (n_reco,)   ← signal in reco-energy bins
    s_new = s_col[:, np.newaxis] * rate   # (n_true, 1) * (n_true, n_reco) → (n_true, n_reco)
    return np.sum(s_new, axis=0)          # sum over true-energy axis → (n_reco,)


## Compute `gen_ns` for the selected channel

In [17]:
print(f"Computing signal integral for channel '{CHANNEL}' over {len(mass)} mass values...")
print(f"Using {N_JOBS} CPU core(s) (N_JOBS={N_JOBS}).")

# ── Step 1: integrate spectrum × effective area over true-energy bins ─────────
s = s_integral()   # shape: (n_energy_bins, n_mass)
print(f"  Signal integral done.  Shape: {s.shape}")

# ── Step 2: convolve with energy-dispersion matrix ────────────────────────────
# Parallelise over mass columns
s_conv_list = Parallel(n_jobs=N_JOBS)(
    delayed(apply_convolution)(s[:, k]) for k in range(len(mass))
)
s_conv = np.column_stack(s_conv_list)  # shape: (n_reco_bins, n_mass)
print(f"  Convolution done.      Shape: {s_conv.shape}")

# ── Step 3: bin into background energy bins ───────────────────────────────────
# Each background bin [E_min, E_max] is mapped onto the nearest reco-energy
# bin edges, and the signal is summed + rescaled to account for partial overlap.
s_final = np.zeros((len(bkg), len(mass)))
for i in range(len(bkg)):
    start = find_inflimit(ienergy_values, bkg[i, 0])
    end   = find_suplimit(fenergy_values, bkg[i, 1])
    # Partial-overlap correction: ratio of background bin width to dispersion bin width
    width_ratio = (bkg[i, 1] - bkg[i, 0]) / (fenergy_values[end] - ienergy_values[start])
    s_final[i, :] = np.sum(s_conv[start:end+1, :], axis=0) * width_ratio

print(f"  Binning done.          Shape: {s_final.shape}")
print()
print("gen_ns interpretation:")
print("  N_signal = sigma_v [cm^3/s] × J_factor [GeV^2/cm^5] × gen_ns")
print("  gen_ns shape: (n_bkg_bins, n_mass)")
print(f"  n_bkg_bins = {s_final.shape[0]},  n_mass = {s_final.shape[1]}")


Computing signal integral for channel 'b' over 62 mass values...
Using -1 CPU core(s) (N_JOBS=-1).
  Signal integral done.  Shape: (160, 62)
  Convolution done.      Shape: (160, 62)
  Binning done.          Shape: (35, 62)

gen_ns interpretation:
  N_signal = sigma_v [cm^3/s] × J_factor [GeV^2/cm^5] × gen_ns
  gen_ns shape: (n_bkg_bins, n_mass)
  n_bkg_bins = 35,  n_mass = 62


In [18]:
# ── Save result ───────────────────────────────────────────────────────────────
outfile = os.path.join(RESULTS_DIR, f'{CHANNEL}.txt')
np.savetxt(outfile, s_final)
print(f"gen_ns saved to: {outfile}")
print(f"  Shape: {s_final.shape}  (rows = background bins, cols = DM mass points)")


gen_ns saved to: results/gen_ns/b.txt
  Shape: (35, 62)  (rows = background bins, cols = DM mass points)
